<div dir="rtl" style="text-align: right;">

#  آشنایی با API وی‌ویت (Weaviate)


<div align="center">
  <img src="images/weaviate.png" alt="RAG Overview" width="10%">
</div>

به آزمایشگاه غیرنمره‌ای وی‌ویت خوش آمدید! همان‌طور که وارد دنیای پایگاه‌داده‌های برداری می‌شوید، متوجه می‌شوید که گزینه‌های مختلفی برای ساخت سیستم‌های RAG وجود دارد. در این دوره، روی [API وی‌ویت](https://weaviate.io/) تمرکز می‌کنیم.

این آزمایشگاه یک مقدمه‌ی عملی برای آشنایی با وی‌ویت است تا برای تمرین بعدی کاملاً آماده باشید. نحوه‌ی کار وی‌ویت، امکاناتش و بهترین شکل استفاده از آن را بررسی می‌کنیم.

بزن بریم!

</div>

In [1]:
from IPython.display import HTML

# راست‌چین کردن سلول‌های متنی (Markdown) در این نوت‌بوک
HTML("""
<style>
  .jp-RenderedMarkdown, .rendered_html, .text_cell_render {
    direction: rtl;
    text-align: right;
  }
</style>
""")


<div dir="rtl" style="text-align: right;">


# فهرست مطالب
- [ ۱ - مقدمه](#1)
  - [ ۱.۱ بارگذاری کتابخانه‌های لازم](#1-1)
  - [ ۱.۲ - کلاینت وی‌ویت](#1-2)
- [ ۲ - پیکربندی پایگاه‌داده](#2)
  - [ ۲.۱ ساخت یک Collection](#2-1)
  - [ ۲.۲ پیکربندی Vectorizer](#2-2)
  - [ ۲.۳ ویژگی‌ها (Properties)](#2-3)
  - [ ۲.۴ افزودن عناصر به یک Collection](#2-4)
- [ ۳ - جستجو روی یک Collection](#3)
  - [ ۳.۱ فیلترها](#3-1)
  - [ ۳.۲ جستجوی معنایی](#3-2)
  - [ ۳.۳ جستجوی BM25](#3-3)
  - [ ۳.۴ جستجوی ترکیبی (Hybrid)](#3-4)
  - [ ۳.۵ بازرتبه‌بندی (Reranking)](#3-5)


</div>

<div dir="rtl" style="text-align: right;">

<a id='1'></a>
## ۱ - مقدمه
---

pip install weaviate-client joblib tqdm openai


<a id='1-1'></a>
### ۱.۱ بارگذاری کتابخانه‌های لازم
سلول زیر را اجرا کنید تا کتابخانه‌های لازم برای این تمرین بارگذاری شوند.

</div>

In [2]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

In [3]:
from utils import (
    suppress_subprocess_output,
    generate_with_single_input,
    generate_embedding,
    print_object_properties,
    llm_rerank,
)

<div dir="rtl" style="text-align: right;">

<a id='1-2'></a>
### ۱.۲ - کلاینت وی‌ویت

برای شروع کار با API وی‌ویت، باید یک `client` بسازید. دو راه اصلی برای اجرای وی‌ویت وجود دارد:

1. **Embedded** (فقط روی لینوکس/مک): وی‌ویت درون همین پردازش پایتون اجرا می‌شود، بدون نیاز به نصب جداگانه.
2. **Docker / سرور مستقل** (روی هر سیستم‌عاملی از جمله ویندوز کار می‌کند): وی‌ویت به‌عنوان یک کانتینر جدا (مثلاً با `docker compose up -d`) اجرا می‌شود و شما با `weaviate.connect_to_local()` به آن وصل می‌شوید.

در این آزمایزگاه فرض می‌کنیم Weaviate با فایل `docker-compose.yml` (که کنار همین نوت‌بوک قرار دارد) از قبل بالا آمده است.

وقتی کلاینت را می‌سازید، باید یک مدل embedding برای بردارسازی معرفی کنید. چون OpenAI یک سرویس پولی است، به‌صورت پیش‌فرض از **Ollama** (کاملاً رایگان و لوکال) استفاده می‌کنیم؛ اما کد طوری نوشته شده که با تغییر یک متغیر می‌توانید به OpenAI هم سوییچ کنید.

مثال اتصال با هدر OpenAI (اختیاری):

```Python
import weaviate
client = weaviate.connect_to_local(
    headers={
        "X-OpenAI-Api-Key": YOUR_OPENAI_API_KEY
    },
)
```

</div>

<div dir="rtl" style="text-align: right;">

حالا کلاینت را بارگذاری می‌کنیم!

تابع `suppress_subprocess_output()` برای خاموش‌کردن لاگ‌های اضافی طراحی شده که ممکن است خروجی آزمایشگاه را شلوغ کنند. این لاگ‌ها در این آزمایشگاه بررسی نمی‌شوند، ولی اگر کنجکاو هستید، آزادید این خط را حذف کنید تا لاگ‌ها را ببینید.

چون از Docker استفاده می‌کنیم، فقط کافیست با `weaviate.connect_to_local()` وصل شویم؛ آدرس پیش‌فرض `localhost:8080` (REST) و `localhost:50051` (gRPC) است — همان پورت‌هایی که در `docker-compose.yml` باز کردیم.

</div>

In [8]:
from dotenv import load_dotenv
load_dotenv(override=True) 

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = "https://api.gapgpt.app/" #os.getenv("OPENAI_BASE_URL")

with suppress_subprocess_output():
    client = weaviate.connect_to_local(
        headers={
            "X-OpenAI-Api-Key": openai_api_key,
            "X-OpenAI-BaseURL": openai_base_url
        }
    )


C:\Users\Alireza\.conda\envs\python3.12\Lib\site-packages\weaviate\warnings.py:312: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\Alireza\AppData\Local\Temp\ipykernel_41320\848829263.py:8: ResourceWarning: unclosed <socket.socket fd=1624, family=23, type=1, proto=0, laddr=('::1', 57410, 0, 0), raddr=('::1', 8080, 0, 0)>
  client = weaviate.connect_to_local(


<div dir="rtl" style="text-align: right;">

با `client` تعریف‌شده، کاربرد اصلی شما ساختن یک Collection، افزودن عناصر به آن و جستجو روی آن است.

</div>

<div dir="rtl" style="text-align: right;">

<a id='2'></a>
## ۲ - پیکربندی پایگاه‌داده

---

در این بخش، مهم‌ترین شیء این آزمایشگاه و تمرین بعدی را بررسی می‌کنیم: [کالکشن (Collection)](https://weaviate.io/developers/weaviate/manage-data/collections) - نامی که وی‌ویت به یک گروه از داده‌ها می‌دهد که برای بازیابی ایندکس می‌شوند. جریان کاری که در جلسات دیدید را به یاد بیاورید:

<div align="center">
  <img src="images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### ۲.۱ ساخت یک Collection

برای ساخت یک Collection، چند پارامتر باید تنظیم شوند. مهم‌ترین‌ها برای هدف ما:

- `name`: نام Collection؛ همین نام برای بارگذاری دوباره‌ی آن استفاده می‌شود.
- `vectorizer_config`: لیستی از پیکربندی‌های vectorizer. می‌توانید بیش از یک پیکربندی بدهید، یعنی در همان پایگاه‌داده‌ی برداری می‌توانید داده‌هایتان را با چند مدل embedding مختلف بردارسازی کنید. در این‌جا فقط از یکی استفاده می‌کنیم.

بیایید یک دیتاست بارگذاری کنیم تا این بخش را نشان دهیم.

</div>

In [9]:
data = joblib.load("data.joblib")
print_object_properties(data[0])


place: دره ستارگان
state: هرمزگان
description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.
best_season_to_visit: پاییز, زمستان
attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
budget: ارزان
user_ratings: 4.6
last_updated: 2025-01-01T00:00:00Z



<div dir="rtl" style="text-align: right;">

دیتاست مجموعه‌ای از مکان‌های گردشگری است، همراه با چند ویژگی که هر مکان را توصیف می‌کند. ویژگی‌ها این‌ها هستند: `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`. هنگام ساخت Collection، باید برای هر کلید این دیکشنری یک Property با نوع داده‌ی مناسب تعریف کنید.

</div>

<div dir="rtl" style="text-align: right;">

<a id='2-2'></a>
### ۲.۲ پیکربندی Vectorizer

همان‌طور که گفته شد، برای بردارسازی داده از یک مدل embedding استفاده می‌کنیم. به‌صورت پیش‌فرض از ماژول بومی وی‌ویت برای **Ollama** (`text2vec-ollama`) استفاده می‌کنیم که کاملاً رایگان و لوکال است. اگر بخواهید، با تغییر متغیر `BACKEND` به `"openai"` می‌توانید از OpenAI هم استفاده کنید (نیاز به `OPENAI_API_KEY` دارد).

می‌توانید لیستی از vectorizerهای مختلف بدهید تا یک Collection چند بردار مختلف برای هر آبجکت داشته باشد، و انتخاب کنید کدام ویژگی‌ها با کدام vectorizer بردار شوند. در این دوره فقط با یک vectorizer کار می‌کنیم.

در این‌جا این ویژگی‌ها بردارسازی می‌شوند:

`place, state, description, best_season_to_visit, attractions, budget`

این ویژگی‌ها به هم چسبانده و سپس بردارسازی می‌شوند. هنگام تعریف هر Property، می‌توانید انتخاب کنید که نام آن ویژگی هم در متن بردارسازی‌شده بیاید یا نه. مثلاً برای `budget` منطقی است که نام ویژگی هم اضافه شود، چون کلمه‌ی تنهای «ارزان» به‌تنهایی اطلاعات کافی نمی‌دهد که منظور از «ارزان» چیست.

</div>

In [11]:
# انتخاب backend: "ollama" (رایگان و لوکال) یا "openai" (نیاز به API key)
BACKEND = "openai"

if BACKEND == "ollama":
    vectorizer_config = [Configure.NamedVectors.text2vec_ollama(
        name="vector",  # نامی که برای دسترسی به بردارهای این Collection لازم دارید
        source_properties=['place', 'state', 'description', 'best_season_to_visit', 'attractions', 'budget'],
        vectorize_collection_name=False,  # نام خود Collection را بردارسازی نکن
        api_endpoint="http://host.docker.internal:11434",  # آدرس Ollama از داخل کانتینر داکر
        model="bge-m3",  # مدل embedding چندزبانه (فارسی را هم خوب پشتیبانی می‌کند)
    )]
else:
    vectorizer_config = [Configure.NamedVectors.text2vec_openai(
        name="vector",
        source_properties=['place', 'state', 'description', 'best_season_to_visit', 'attractions', 'budget'],
        vectorize_collection_name=False,
        model="text-embedding-3-small",
    )]


<div dir="rtl" style="text-align: right;">

<a id='2-3'></a>
### ۲.۳ ویژگی‌ها (Properties)

در یک Collection، ویژگی‌های هر داده را Property می‌نامند.

</div>

In [12]:
# اگر Collection از قبل وجود دارد، حذفش می‌کنیم
if client.collections.exists("example_collection_openai"):
    client.collections.delete("example_collection_openai")


In [13]:
if not client.collections.exists('example_collection_openai'):  # فقط اگر وجود نداشت بساز
    collection = client.collections.create(
            name='example_collection_openai',
            vectorizer_config=vectorizer_config,  # پیکربندی‌ای که قبلاً تعریف کردیم

            properties=[  # تعریف ویژگی‌ها
            Property(name="place", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="state", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="best_season_to_visit", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="attractions", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="budget", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="user_ratings", data_type=DataType.NUMBER),
            Property(name="last_updated", data_type=DataType.DATE),

        ]
        )
else:
    collection = client.collections.get("example_collection")


C:\Users\Alireza\.conda\envs\python3.12\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


<div dir="rtl" style="text-align: right;">

اجرای این سلول یک Collection می‌سازد و آن را برمی‌گرداند. چاپ کردنش، پیکربندی Collection را نشان می‌دهد.

</div>

In [14]:
print(collection)

<weaviate.Collection config={
  "name": "Example_collection_openai",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    },
    "stopword_presets": null
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "place",
      "description": null,
      "data_type": "text",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": true,
      "nested_properties": null,
      "text_analyzer": null,
      "tokenization": "word",
      "vectorizer_config": null,
      "vectorizer": null,
      "vectorizer_configs

<div dir="rtl" style="text-align: right;">

اگر بخواهید Collection‌ای بسازید که از قبل وجود دارد، یک exception پرتاب می‌شود:

</div>

In [15]:
try:
    collection = client.collections.create(
        name='example_collection_openai',

        vectorizer_config=vectorizer_config,  # پیکربندی‌ای که قبلاً تعریف کردیم

        properties=[  # تعریف ویژگی‌ها
        Property(name="place", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="state", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="best_season_to_visit", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="attractions", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="budget", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="user_ratings", data_type=DataType.NUMBER),
        Property(name="last_updated", data_type=DataType.DATE),

    ]
    )
except Exception as e:
    print(e)


Collection may not have been created properly.! Unexpected status code: 422, with response body: {'allowed': None, 'error': [{'message': 'class name Example_collection_openai already exists'}]}.


<div dir="rtl" style="text-align: right;">

همچنین می‌توانید همه‌ی Collectionهای ذخیره‌شده را بازیابی کنید:

</div>

In [16]:
client.collections.list_all().keys()

dict_keys(['Example_collection', 'Example_collection_openai'])

<div dir="rtl" style="text-align: right;">

نتیجه‌ی `.list_all()` یک دیکشنری است که کلیدهایش نام Collectionها و مقادیرش ویژگی‌های آن‌هاست.

</div>

<div dir="rtl" style="text-align: right;">

<a id='2-4'></a>
### ۲.۴ افزودن عناصر به یک Collection

وقتی یک Collection می‌سازید، یک Collection خالی دارید. حالا باید عناصری به آن اضافه کنید. وقتی یک عنصر اضافه می‌کنید، دو اتفاق مهم در پس‌زمینه می‌افتد:

۱. اطلاعات بردارسازی می‌شوند (طبق پیکربندی Collection)

۲. ایندکس HNSW برای بهینه‌سازی جستجو به‌روزرسانی می‌شود (همان‌طور که در جلسات دیدید). این کار پشت‌صحنه انجام می‌شود و نمی‌بینیدش، ولی ممکن است باعث شود فرآیند کمی طول بکشد

افزودن عناصر با `collection.batch` انجام می‌شود که امکانات مفید بیشتری هم دارد. مثلاً می‌توانید مشخص کنید در هر batch چند آبجکت ارسال شود، خطاها در حین ایمپورت را مدیریت کنید، و با کاهش تعداد فراخوانی‌های شبکه، کارایی را بهتر کنید. در این مثال، هر بار فقط یک عنصر با یک درخواست هم‌زمان ارسال می‌شود.

می‌توانید به هر عنصری که اضافه می‌کنید یک uuid (شناسه‌ی یکتا) بدهید، تا از ورود تکراری داده جلوگیری شود.

بیایید در عمل ببینیم!

</div>

In [17]:
# راه‌اندازی یک فرآیند batch با اندازه‌ی ثابت و هم‌زمانی مشخص
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    # پیمایش روی زیرمجموعه‌ای از دیتاست
    for document in tqdm(data):  # tqdm کتابخانه‌ای برای نمایش نوار پیشرفت است
            # ساخت یک UUID بر اساس محتوای هر رکورد برای شناسایی یکتا
            uuid = generate_uuid5(document)

            # افزودن آبجکت به batch همراه با ویژگی‌ها و UUID
            # properties یک دیکشنری می‌خواهد که کلیدهایش همان Propertyها هستند
            batch.add_object(
                properties=document,
                uuid=uuid,
            )


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:19<00:00,  3.95s/it]


<div dir="rtl" style="text-align: right;">

عالی! حالا یک Collection پر از بردار دارید! می‌توانید تعداد بردارها را با `len(collection)` بررسی کنید:

</div>

In [18]:
len(collection)

20

<div dir="rtl" style="text-align: right;">

<a id='3'></a>
## ۳ - جستجو روی یک Collection

در این بخش، یاد می‌گیرید چطور روی یک Collection جستجو کنید. می‌توانید:

- روی متادیتا جستجو کنید
- با جستجوی معنایی جستجو کنید
- با BM25 جستجو کنید
- با فیلتر جستجو کنید

بیایید چند مثال ببینیم.

</div>

<div dir="rtl" style="text-align: right;">

<a id='3-1'></a>
### ۳.۱ فیلترها

قبل از ورود به جستجو، بیایید فیلترها را بشناسیم. فیلترها روشی برای محدودکردن جستجو بر اساس معیارهای خاص هستند و بسیار انعطاف‌پذیرند. معمولاً آن‌ها را به‌عنوان یک آرگومان در یک کوئری پاس می‌دهید. بیایید یک مثال ببینیم.

</div>

In [ ]:
# این‌جا ۲ آبجکت را با فیلتر روی ویژگی 'user_ratings' می‌گیریم، فقط آبجکت‌هایی با مقدار بزرگ‌تر یا مساوی ۳.۵
result = collection.query.fetch_objects(limit=2, filters=Filter.by_property('user_ratings').greater_or_equal(3.5))


<div dir="rtl" style="text-align: right;">

نتیجه یک آبجکت به اسم QueryReturn است:

</div>

In [ ]:
result

<div dir="rtl" style="text-align: right;">

می‌توانید به آبجکت‌هایش با `result.objects` دسترسی داشته باشید

</div>

In [ ]:
result.objects

<div dir="rtl" style="text-align: right;">

پس هر عنصر در این لیست، یک عنصر از Collection است.

</div>

In [ ]:
obj = result.objects[0]

<div dir="rtl" style="text-align: right;">

می‌توانید ویژگی‌های هر آبجکت را بررسی کنید، که یک دیکشنری است.

</div>

In [ ]:
obj.properties

<div dir="rtl" style="text-align: right;">

در این دوره، روش فیلتر کردن `.by_property` است. مثال‌های بیشتری از فیلتر را با معرفی سایر متدهای کوئری خواهید دید.

</div>

<div dir="rtl" style="text-align: right;">

<a id='3-2'></a>
### ۳.۲ جستجوی معنایی

می‌توانید از جستجوی معنایی برای جستجو روی Collection استفاده کنید. این روش از بردارها برای محاسبه‌ی فاصله بین آن‌ها استفاده می‌کند و نزدیک‌ترین‌ها را برمی‌گرداند. باید یک کوئری بدهید که بردارسازی و سپس با عناصر Collection مقایسه می‌شود. متد آن `.near_text` است.

</div>

In [ ]:
result = collection.query.near_text(query = 'می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم.', limit = 4)


In [ ]:
# بیایید روی آبجکت‌های نتیجه پیمایش کنیم و ویژگی‌هایشان را چاپ کنیم
for obj in result.objects:
    print_object_properties(obj.properties)


<div dir="rtl" style="text-align: right;">

می‌توانید همچنین روی عناصری که `budget = ارزان` دارند هم جستجو کنید:

</div>

In [ ]:
result = collection.query.near_text(query = 'می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم.',
                                    filters = Filter.by_property('budget').equal('ارزان'),
                                    limit = 4)


In [ ]:
# بیایید روی آبجکت‌های نتیجه پیمایش کنیم و ویژگی‌هایشان را چاپ کنیم
for obj in result.objects:
    print_object_properties(obj.properties)


<div dir="rtl" style="text-align: right;">

همچنین می‌توانید یک لیست از مقادیر ممکن به یک فیلتر بدهید، با استفاده از `.contains_any`:

</div>

In [ ]:
result = collection.query.near_text(query = 'می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم.',
                                    filters = Filter.by_property('budget').contains_any(['ارزان','متوسط']),
                                    limit = 4)


In [ ]:
# بیایید روی آبجکت‌های نتیجه پیمایش کنیم و ویژگی‌هایشان را چاپ کنیم
for obj in result.objects:
    print_object_properties(obj.properties)


<div dir="rtl" style="text-align: right;">

<a id='3-3'></a>
### ۳.۳ جستجوی BM25

برای انجام جستجوی BM25، کافیست `collection.query.bm25` را اجرا کنید؛ پارامترهای معمول `query`، `limit` و `filters` قابل استفاده‌اند.

</div>

In [ ]:
result = collection.query.bm25(query = 'می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم.',
                                    filters = Filter.by_property('budget').contains_any(['ارزان','متوسط']),
                                    limit = 4)


In [ ]:
# بیایید روی آبجکت‌های نتیجه پیمایش کنیم و ویژگی‌هایشان را چاپ کنیم
for obj in result.objects:
    print_object_properties(obj.properties)


<div dir="rtl" style="text-align: right;">

<a id='3-4'></a>
### ۳.۴ جستجوی ترکیبی (Hybrid)

این همان جستجوی RRF است که در جلسات دیدید. علاوه بر پارامترهای معمول کوئری، می‌توانید یک `alpha` بدهید تا مشخص کنید چه میزان از BM25 در ترکیب دخیل باشد.

</div>

In [ ]:
result = collection.query.hybrid(query = 'می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان هستم.',
                                    filters = Filter.by_property('budget').contains_any(['ارزان','متوسط']),
                                    alpha = 0.3,
                                    limit = 4)


In [ ]:
# بیایید روی آبجکت‌های نتیجه پیمایش کنیم و ویژگی‌هایشان را چاپ کنیم
for obj in result.objects:
    print_object_properties(obj.properties)


<div dir="rtl" style="text-align: right;">

<a id='3-5'></a>
### ۳.۵ بازرتبه‌بندی (Reranking)

نسخه‌ی اصلی وی‌ویت امکان بازرتبه‌بندی داخلی (`Rerank`) را دارد، اما این قابلیت به یک ماژول reranker جداگانه نیاز دارد (مثل `reranker-cohere`، `reranker-voyageai` یا نسخه‌ی self-hosted `reranker-transformers` که نیاز به دانلود یک مدل سنگین و اجرای یک سرور جدا دارد).

چون در این‌جا فقط از Ollama/OpenAI استفاده می‌کنیم، به‌جای ماژول داخلی وی‌ویت، بازرتبه‌بندی را **در کد پایتون خودمان** با کمک همان مدل زبانی (LLM-as-reranker) انجام می‌دهیم: از مدل چت می‌خواهیم به هر کاندید یک نمره‌ی ربط بدهد، و بر اساس آن نمره مرتب می‌کنیم. تابع آماده‌ی این کار (`llm_rerank`) در `utils.py` تعریف شده است.

</div>

In [ ]:
# ابتدا با جستجوی معنایی چند کاندید می‌گیریم
response = collection.query.near_text(
    query="می‌خواهم برای سفر زمستانی پیشنهاد بگیرم. به دنبال مکان‌های ارزان و سرگرم‌کننده هستم.",
    limit=5,
)
candidates = [obj.properties for obj in response.objects]

# سپس با کمک مدل زبانی (Ollama یا OpenAI) بر اساس ویژگی 'attractions' بازرتبه‌بندی می‌کنیم
reranked = llm_rerank(query="مکان‌های سرگرم‌کننده", candidates=candidates, text_key="attractions")


In [ ]:
# بیایید روی نتایج بازرتبه‌بندی‌شده پیمایش کنیم و امتیاز هرکدام را ببینیم
for item in reranked:
    print(f"امتیاز رتبه‌بندی: {item['_rerank_score']}")
    print_object_properties({k: v for k, v in item.items() if k != '_rerank_score'})
    print()


In [ ]:
# فراموش نکنید کلاینت را ببندید!
client.close()


<div dir="rtl" style="text-align: right;">

آفرین! شما مبانی API وی‌ویت را که در این دوره استفاده خواهید کرد، تمام کردید!

</div>